In [ ]:
import os
import requests
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# ============= НАСТРОЙКИ =============
YEAR = 2014
BASE_URL = f"https://data.gats-inc.com/saber/Version2_0/Level2A/{YEAR}"
SAVE_ROOT = f"G:/SABER_L2A/{YEAR}"

# Начальный номер файла для исследуемого года
START_FILE_NUM = 67912

# Количество файлов в первом дне (001) и в остальных
FILES_IN_FIRST_DAY = 14
FILES_IN_OTHER_DAYS = 15
TOTAL_DAYS = 305  

# Количество потоков для параллельной загрузки 
MAX_WORKERS = 5

def download_file(url, save_path):
    """Скачивает один файл и показывает прогресс"""
    try:
        if os.path.exists(save_path):
            if os.path.getsize(save_path) > 1024: 
                return f"Пропущен (уже есть): {os.path.basename(save_path)}"
        response = requests.get(url, stream=True, timeout=30)
        response.raise_for_status()
        total_size = int(response.headers.get('content-length', 0))
        with open(save_path, 'wb') as f:
            with tqdm(total=total_size, unit='B', unit_scale=True, 
                     desc=os.path.basename(save_path), leave=False) as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
                    pbar.update(len(chunk))
        return f"Успешно: {os.path.basename(save_path)}"
    except Exception as e:
        return f"Ошибка {os.path.basename(save_path)}: {str(e)}"
def generate_file_list():
    """Генерирует список всех URL и путей для сохранения"""
    file_tasks = []
    current_file_num = START_FILE_NUM
    for day in range(172, TOTAL_DAYS + 1):
        day_folder = f"{day:03d}"
        save_dir = os.path.join(SAVE_ROOT, day_folder)
        os.makedirs(save_dir, exist_ok=True)
        print(f"\nПроверка дня {day_folder}...")
        first_file_in_day = None
        search_start = current_file_num
        for offset in range(-5, 20):  
            test_num = search_start + offset
            filename = f"SABER_L2A_{YEAR}{day_folder}_{test_num}_02.07.nc"
            url = f"{BASE_URL}/{day_folder}/{filename}"
            try:
                response = requests.head(url, timeout=5)
                if response.status_code == 200:
                    first_file_in_day = test_num
                    print(f"  Первый файл: {filename}")
                    break
            except:
                continue
        if first_file_in_day is None:
            print(f"  Не удалось найти файлы для дня {day_folder}, пропускаем")
            continue
        file_num = first_file_in_day
        files_found = 0
        max_files_in_day = 20
        while files_found < max_files_in_day:
            filename = f"SABER_L2A_{YEAR}{day_folder}_{file_num}_02.07.nc"
            url = f"{BASE_URL}/{day_folder}/{filename}"
            save_path = os.path.join(save_dir, filename)
            try:
                response = requests.head(url, timeout=5)
                if response.status_code == 200:
                    print(f"    Найден: {filename}")
                    download_result = download_file(url, save_path)
                    print(f"      {download_result}")
                    files_found += 1
                    current_file_num = file_num + 1
                    file_num += 1
                else:
                    if files_found > 0:
                        print(f"    Достигнут конец дня после {files_found} файлов")
                        break
                    else:
                        file_num += 1
            except Exception as e:
                print(f"    Ошибка проверки {filename}: {e}")
                file_num += 1
        print(f"  День {day_folder}: найдено {files_found} файлов")
    return file_tasks
def main():
    print(f"Начинаю подготовку к загрузке данных SABER за {YEAR} год...")
    print(f"Файлы будут сохранены в: {SAVE_ROOT}")
    file_tasks = generate_file_list()
    total_files = len(file_tasks)
    print(f"Всего файлов для загрузки: {total_files}")
    successful = 0
    failed = 0
    skipped = 0
    print(f"\nНачинаю загрузку ({MAX_WORKERS} потоков)...")
    start_time = time.time()
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_file = {executor.submit(download_file, url, path): (url, path) 
                         for url, path in file_tasks}
        with tqdm(total=total_files, desc="Общий прогресс", unit="файл") as pbar:
            for future in as_completed(future_to_file):
                result = future.result()
                pbar.update(1)
                
                if "Успешно" in result:
                    successful += 1
                elif "Пропущен" in result:
                    skipped += 1
                else:
                    failed += 1
                    print(f"\n{result}")  
    
    elapsed_time = time.time() - start_time
    
    # Итоговая статистика
    print(f"\n{'='*60}")
    print(f"ЗАГРУЗКА ЗАВЕРШЕНА!")
    print(f"{'='*60}")
    print(f"Время выполнения: {elapsed_time/60:.1f} минут")
    print(f"Всего файлов: {total_files}")
    print(f"Успешно загружено: {successful}")
    print(f"Пропущено (уже есть): {skipped}")
    print(f"Ошибок: {failed}")
    print(f"{'='*60}")
    print(f"Данные сохранены в: {SAVE_ROOT}")

if __name__ == "__main__":
    main()

Начинаю подготовку к загрузке данных SABER за 2014 год...
Файлы будут сохранены в: G:/SABER_L2A/2014

Проверка дня 172...
  Первый файл: SABER_L2A_2014172_67912_02.07.nc
    Найден: SABER_L2A_2014172_67912_02.07.nc
      Пропущен (уже есть): SABER_L2A_2014172_67912_02.07.nc
    Найден: SABER_L2A_2014172_67913_02.07.nc
      Пропущен (уже есть): SABER_L2A_2014172_67913_02.07.nc
    Найден: SABER_L2A_2014172_67914_02.07.nc
      Пропущен (уже есть): SABER_L2A_2014172_67914_02.07.nc
    Найден: SABER_L2A_2014172_67915_02.07.nc
      Пропущен (уже есть): SABER_L2A_2014172_67915_02.07.nc
    Найден: SABER_L2A_2014172_67916_02.07.nc
      Пропущен (уже есть): SABER_L2A_2014172_67916_02.07.nc
    Найден: SABER_L2A_2014172_67917_02.07.nc
      Пропущен (уже есть): SABER_L2A_2014172_67917_02.07.nc
    Найден: SABER_L2A_2014172_67918_02.07.nc
      Пропущен (уже есть): SABER_L2A_2014172_67918_02.07.nc
    Найден: SABER_L2A_2014172_67919_02.07.nc
      Пропущен (уже есть): SABER_L2A_2014172_67919

      Успешно: SABER_L2A_2014174_67941_02.07.nc
    Найден: SABER_L2A_2014174_67942_02.07.nc


      Успешно: SABER_L2A_2014174_67942_02.07.nc
    Найден: SABER_L2A_2014174_67943_02.07.nc


      Успешно: SABER_L2A_2014174_67943_02.07.nc
    Найден: SABER_L2A_2014174_67944_02.07.nc


      Успешно: SABER_L2A_2014174_67944_02.07.nc
    Найден: SABER_L2A_2014174_67945_02.07.nc


...

      Успешно: SABER_L2A_2014196_68269_02.07.nc
    Найден: SABER_L2A_2014196_68270_02.07.nc


      Успешно: SABER_L2A_2014196_68270_02.07.nc
    Найден: SABER_L2A_2014196_68271_02.07.nc


      Успешно: SABER_L2A_2014196_68271_02.07.nc
    Найден: SABER_L2A_2014196_68272_02.07.nc


      Успешно: SABER_L2A_2014196_68272_02.07.nc
    Найден: SABER_L2A_2014196_68273_02.07.nc


      Ошибка SABER_L2A_2014196_68273_02.07.nc: ("Connection broken: ConnectionResetError(10054, 'Удаленный хост принудительно разорвал существующее подключение', None, 10054, None)", ConnectionResetError(10054, 'Удаленный хост принудительно разорвал существующее подключение', None, 10054, None))
    Ошибка проверки SABER_L2A_2014196_68274_02.07.nc: HTTPSConnectionPool(host='data.gats-inc.com', port=443): Max retries exceeded with url: /saber/Version2_0/Level2A/2014/196/SABER_L2A_2014196_68274_02.07.nc (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002200646DE50>: Failed to resolve 'data.gats-inc.com' ([Errno 11004] getaddrinfo failed)"))
    Ошибка проверки SABER_L2A_2014196_68275_02.07.nc: HTTPSConnectionPool(host='data.gats-inc.com', port=443): Max retries exceeded with url: /saber/Version2_0/Level2A/2014/196/SABER_L2A_2014196_68275_02.07.nc (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002200646D310>: Failed to